# Ford used cars — price regression

Used **Ford** listings: mix of **model**, **year**, **mileage**, **transmission**, **fuel type**, **tax**, **mpg**, **engine size**, with **price** as the numeric target. This notebook is structured as **EDA → encode categoricals → scale selected numerics → train/test split → linear regression → error metrics** (including **R²** and **adjusted R²**).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")


In [ ]:
df = pd.read_csv("ford.csv")


## First look at the table

In [ ]:
df.head()


In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df.isnull().sum()


## Plots for EDA

**Histogram** of price shows skew and typical range. The **heatmap** is Pearson correlation between *numeric* columns only (categories are not in the matrix until encoded). **Box plots** and **scatter** relate price (or mpg) to other fields—useful for outliers and non-linear patterns a straight-line model might miss.

In [ ]:
sns.histplot(df["price"], bins=50, kde=True)
plt.title("Distribution of price")
plt.show()


In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="RdYlGn", fmt=".2f")
plt.title("Numeric correlations")
plt.show()


In [ ]:
sns.boxplot(data=df, x="year", y="price")
plt.title("Price by year")
plt.show()


In [ ]:
sns.scatterplot(data=df, x="mileage", y="price", alpha=0.3)
plt.title("Price vs mileage")
plt.show()


In [ ]:
sns.boxplot(data=df, x="engineSize", y="price")
plt.title("Price by engine size")
plt.show()


In [ ]:
df.columns.tolist()


In [ ]:
sns.boxplot(data=df, x="transmission", y="price")
plt.xticks(rotation=20)
plt.title("Price by transmission")
plt.show()


In [ ]:
sns.boxplot(data=df, x="fuelType", y="price")
plt.xticks(rotation=20)
plt.title("Price by fuel type")
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(x=df["model"], y=df["price"])
plt.xticks(rotation=45, ha="right")
plt.title("Price by model")
plt.tight_layout()
plt.show()


In [ ]:
sns.boxplot(data=df, x="tax", y="mpg")
plt.title("MPG by road tax band")
plt.show()


## Features and target

**Supervised regression:** predict **`price`** from the other columns. `X` holds predictors; `y` is the target vector.

In [ ]:
X = df.drop("price", axis=1)
y = df["price"]
X.shape, y.shape


## One-hot encoding (`get_dummies`)

**Nominal** columns (`model`, `fuelType`, `transmission`) have no natural order. **`pd.get_dummies`** expands them into 0/1 columns. **`drop_first=True`** removes one reference category per original column to reduce redundancy (multicollinearity) when using a linear model with an intercept.

In [ ]:
X_encoded = pd.get_dummies(
    X, columns=["model", "fuelType", "transmission"], drop_first=True
)
X_encoded = X_encoded.astype(int)
X_encoded.head()


## Standardize selected numerics only

**`StandardScaler`** puts **`mileage`**, **`engineSize`**, **`tax`**, **`mpg`**, and **`year`** on a comparable scale. **Dummy** columns stay 0/1 (do not scale them: their scale is already meaningful). In production you would **`fit`** the scaler on **training** data only, then **`transform`** test data; here we fit on the full feature matrix for simplicity—see the Core ML README for the leakage note.

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = ["mileage", "engineSize", "tax", "mpg", "year"]
scaler = StandardScaler()
X_encoded[numerical_cols] = scaler.fit_transform(X_encoded[numerical_cols])
X_encoded.head()


## Train–test split and linear regression

**`train_test_split`** holds out a random fraction of rows (**`test_size`**) for evaluation so you do not judge the model on the same rows it learned from. **`LinearRegression`** fits coefficients that minimize squared error (ordinary least squares). **`random_state`** makes splits reproducible.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.33, random_state=42
)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred[:10]


## Metrics on the held-out set

In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
n = X_test.shape[0]
p = X_test.shape[1]
adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))

print(f"MSE:  {mse:,.0f}")
print(f"RMSE: {rmse:,.0f}")
print(f"MAE:  {mae:,.0f}")
print(f"R²:   {r2:.4f}")
print(f"Adj R²: {adjusted_r2:.4f}")


### What these metrics mean

- **MSE / RMSE**: average squared error; **RMSE** is in the same units as **price** (pounds), so it is easy to interpret as a typical error magnitude.
- **MAE**: mean absolute error—less sensitive to huge outliers than MSE.
- **R²**: fraction of variance in **`y_test`** explained by the model (1 is perfect, 0 is no better than predicting the mean; negative means worse than the mean).
- **Adjusted R²**: penalizes extra predictors so adding useless columns does not automatically inflate “goodness of fit.”